# Task #67 — Artefact tra cứu cho tiền xử lý API (`src/api`)

`POST /predict` (Task #66) chỉ nhận `customer_zip_code_prefix` + `primary_seller_zip_code_prefix` (quyết định đã thống nhất với User) thay vì bắt client tự gửi `state` hay khoảng cách đã tính sẵn. Notebook này dựng 3 artefact nhỏ để Task #67 (hàm tiền xử lý) tra cứu, tái sử dụng đúng logic đã dùng khi huấn luyện model (notebook 23):

1. `models/zip_state_lookup.json` — `customer_zip_code_prefix` → `customer_state` (duy nhất 100%), và `primary_seller_zip_code_prefix` → `primary_seller_state` (17/2246 zip ánh xạ >1 state — quyết định tie-break đã thống nhất với User: đa số phiếu, hòa → alphabet đầu).
2. `models/zip_geo_lookup.json` — `zip_code_prefix` → `[lat, lng]` trung vị (`geo_agg`, y hệt notebook 23).
3. Thêm `train_median_distance_km` vào `models/final_model.json` — giá trị trung vị khoảng cách của **tập train** dùng để điền khi 1 trong 2 zip không tra được toạ độ (notebook 23 đã dùng giá trị này để điền 385/77.156 dòng train + 91/19.289 dòng test, in ra `434.70 km` nhưng chưa từng lưu lại full precision ở đâu — notebook này tính lại và lưu lại).


## 1. Hàm khoảng cách haversine (y hệt notebook 23)


In [1]:
import json

import numpy as np
import pandas as pd

RAW = "../data/raw"
PROCESSED = "../data/processed"
MODELS = "../models"


def haversine_km(lat1, lng1, lat2, lng2):
    lat1, lng1, lat2, lng2 = map(np.radians, [lat1, lng1, lat2, lng2])
    dlat = lat2 - lat1
    dlng = lng2 - lng1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlng / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(a))


## 2. `customer_zip_code_prefix` → `customer_state`

Đã xác nhận 100% duy nhất (không ambiguous) khi kiểm tra ở phiên trước — lấy giá trị đầu tiên mỗi nhóm.


In [2]:
customers = pd.read_csv(
    f"{RAW}/olist_customers_dataset.csv",
    usecols=["customer_zip_code_prefix", "customer_state"],
)
customer_state_by_zip = (
    customers.groupby("customer_zip_code_prefix")["customer_state"].first().to_dict()
)
print("customer zip prefixes:", len(customer_state_by_zip))


customer zip prefixes: 14994


## 3. `primary_seller_zip_code_prefix` → `primary_seller_state`

17/2246 zip prefix ánh xạ >1 state trong `olist_sellers_dataset.csv`. Tie-break đã thống nhất với User: chọn state xuất hiện nhiều nhất (đa số phiếu); nếu hòa phiếu, chọn theo alphabet đầu.


In [3]:
sellers = pd.read_csv(
    f"{RAW}/olist_sellers_dataset.csv",
    usecols=["seller_zip_code_prefix", "seller_state"],
)

seller_state_by_zip = {}
ambiguous_count = 0
for zip_prefix, group in sellers.groupby("seller_zip_code_prefix")["seller_state"]:
    counts = group.value_counts()
    top_count = counts.max()
    tied_candidates = sorted(counts[counts == top_count].index)
    seller_state_by_zip[zip_prefix] = tied_candidates[0]
    if len(counts) > 1:
        ambiguous_count += 1

print("seller zip prefixes:", len(seller_state_by_zip), " ambiguous (tie-break applied):", ambiguous_count)


seller zip prefixes: 2246  ambiguous (tie-break applied): 17


## 4. `zip_code_prefix` → toạ độ trung vị (`geo_agg`)

Y hệt logic notebook 23: gộp `olist_geolocation_dataset.csv` về 1 dòng/prefix bằng median lat/lng.


In [4]:
geo = pd.read_csv(f"{RAW}/olist_geolocation_dataset.csv")
geo_agg = (
    geo.groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]]
    .median()
    .reset_index()
)
geo_lookup = {
    int(row.geolocation_zip_code_prefix): [row.geolocation_lat, row.geolocation_lng]
    for row in geo_agg.itertuples()
}
print("geo lookup zip prefixes:", len(geo_lookup))


geo lookup zip prefixes: 19015


## 5. Tính lại `train_median_distance_km`, đối chiếu với dữ liệu đã commit

Tái tạo đúng khoảng cách cho toàn bộ đơn hàng từ 2 zip (`orders_joined.csv`), tách riêng các dòng thuộc tập train (theo `order_id` trong `orders_features_train.csv`), lấy median trong số các dòng **tính được** khoảng cách (trước khi điền) — đúng quy trình notebook 23 đã làm. Đối chiếu 2 chiều: (a) so khoảng cách tính lại với cột `seller_customer_distance_km` đã commit cho các dòng không bị điền — phải khớp tuyệt đối; (b) số dòng thiếu trước khi điền phải khớp đúng 385 (notebook 23 đã báo).


In [5]:
orders = pd.read_csv(
    f"{PROCESSED}/orders_joined.csv",
    usecols=["order_id", "customer_zip_code_prefix", "primary_seller_zip_code_prefix"],
)

distance = (
    orders
    .merge(geo_agg, left_on="customer_zip_code_prefix", right_on="geolocation_zip_code_prefix", how="left")
    .rename(columns={"geolocation_lat": "customer_lat", "geolocation_lng": "customer_lng"})
    .drop(columns="geolocation_zip_code_prefix")
    .merge(geo_agg, left_on="primary_seller_zip_code_prefix", right_on="geolocation_zip_code_prefix", how="left")
    .rename(columns={"geolocation_lat": "seller_lat", "geolocation_lng": "seller_lng"})
    .drop(columns="geolocation_zip_code_prefix")
)
distance["seller_customer_distance_km"] = haversine_km(
    distance["customer_lat"], distance["customer_lng"],
    distance["seller_lat"], distance["seller_lng"],
)
distance = distance[["order_id", "seller_customer_distance_km"]]

train_ids = pd.read_csv(f"{PROCESSED}/orders_features_train.csv", usecols=["order_id"])
train_distance = train_ids.merge(distance, on="order_id", how="left")
train_missing = train_distance["seller_customer_distance_km"].isna().sum()
train_median_distance_km = train_distance["seller_customer_distance_km"].median()
print(f"thieu truoc khi dien (ky vong 385): {train_missing}")
print(f"train_median_distance_km (ky vong ~434.70): {train_median_distance_km}")


thieu truoc khi dien (ky vong 385): 385
train_median_distance_km (ky vong ~434.70): 434.7008425193799


In [6]:
train_committed = pd.read_csv(
    f"{PROCESSED}/orders_features_train.csv", usecols=["order_id", "seller_customer_distance_km"]
)
compare = train_committed.merge(distance, on="order_id", suffixes=("_committed", "_recomputed"))
known_mask = ~train_distance.set_index("order_id")["seller_customer_distance_km"].isna().reindex(compare["order_id"]).values
mismatch = ~np.isclose(
    compare.loc[known_mask, "seller_customer_distance_km_committed"],
    compare.loc[known_mask, "seller_customer_distance_km_recomputed"],
    atol=0.01,
)
print("so dong co the tinh lai khoang cach:", known_mask.sum())
print("so dong lech so voi du lieu da commit (ky vong 0):", mismatch.sum())
assert mismatch.sum() == 0, "Khoang cach tinh lai khong khop du lieu da commit!"
assert train_missing == 385, "So dong thieu khong khop notebook 23!"


so dong co the tinh lai khoang cach: 76771
so dong lech so voi du lieu da commit (ky vong 0): 0


## 6. Lưu artefact

`models/zip_state_lookup.json`, `models/zip_geo_lookup.json` (2 file mới), và thêm 1 key `train_median_distance_km` vào `models/final_model.json` (chỉ thêm, không đổi các key khác).


In [7]:
with open(f"{MODELS}/zip_state_lookup.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "customer_state_by_zip": {str(k): v for k, v in customer_state_by_zip.items()},
            "primary_seller_state_by_zip": {str(k): v for k, v in seller_state_by_zip.items()},
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

with open(f"{MODELS}/zip_geo_lookup.json", "w", encoding="utf-8") as f:
    json.dump({str(k): v for k, v in geo_lookup.items()}, f, indent=2)

with open(f"{MODELS}/final_model.json", "r", encoding="utf-8") as f:
    final_model = json.load(f)
final_model["train_median_distance_km"] = train_median_distance_km
with open(f"{MODELS}/final_model.json", "w", encoding="utf-8") as f:
    json.dump(final_model, f, ensure_ascii=False, indent=2)

print("Da luu 2 file moi + cap nhat final_model.json")


Da luu 2 file moi + cap nhat final_model.json
